# Tin tuyển dụng IT Việt Nam — phân tích từ CSDL

**Mục tiêu:** mô tả cơ cấu tin tuyển dụng, công nghệ, cấp bậc, lương công khai và khác biệt giữa nguồn.

**Nguồn:** [vietnam-it-job-trends](https://github.com/SonPhatTranDeveloper/vietnam-it-job-trends), commit `ad40de27b52f2f222c703e5b55f375d53f46d814`.
Kế thừa file JSONL đã xử lý; không chạy crawler hay AI. Đơn vị phân tích là **tin đăng theo nguồn**, chưa phải số việc làm duy nhất.

Các câu hỏi:
1. Cơ cấu nguồn, nhóm nghề và địa điểm thế nào?
2. Kỹ năng nào phổ biến và khác nhau theo nhóm nghề ra sao?
3. Cơ cấu cấp bậc và mô hình làm việc thế nào?
4. Lương đủ điều kiện so sánh phân bố thế nào?
5. Chất lượng dữ liệu khác nhau thế nào giữa nguồn?

Chạy `python -m joblab.pipeline` trước. Notebook chỉ đọc CSDL; không nhập lại dữ liệu.

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / 'joblab').exists():
    root = root.parent
sys.path.insert(0, str(root))
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display, Markdown
from joblab.analytics import load, summary, skill_counts, quality, insights
px.defaults.template = 'plotly_white'
px.defaults.color_discrete_sequence = ['#24477B', '#007F80', '#D58A2F', '#775A9E']
pio.renderers.default = 'plotly_mimetype'
df, tech, version, candidates, issues = load()
display(pd.Series(version, name='Dataset provenance'))
display(pd.Series(summary(df), name='KPI'))
assert df.posting_id.is_unique


id              ed28b9181a08d53d2625ee291e28cd9f64b08d21fb5613...
input_sha256    64bee64c2a53ba2298b2d47063539b004346b2f21599d1...
rule_version                                                1.0.0
imported_at                      2026-11-10T13:46:23.360101+00:00
input_name                                   published_jobs.jsonl
row_count                                                    3469
Name: Dataset provenance, dtype: object

postings                 3469.000000
companies                1386.000000
salary_disclosure_pct      23.666763
remote_all_pct              3.113289
known_work_model         2705.000000
salary_monthly_n           56.000000
Name: KPI, dtype: float64

## 1. Kiểm tra đầu vào và kết quả chuẩn hóa
Các ô sau đọc Star Schema qua `v_star_postings`: một dòng fact cho một tin trong một phiên bản. Lõi 3NF được giữ để quản lý dữ liệu; kỹ năng nối qua bảng cầu riêng. Bản JSON nguyên trạng được lưu riêng tại `raw_record` và file đầu vào. Nhãn AI kế thừa không được xem là nhãn kiểm chứng.
Không dùng thời gian nhập file thay cho thời gian thu thập.

In [2]:
display(df.head())
display(df.dtypes.to_frame('dtype'))
display(issues)
display(pd.read_csv(root / 'artifacts/before_after.csv').head(12))

,posting_id,snapshot_id,version_id,source,company,title,title_cleaned,role,seniority,location,...,currency,pay_period,gross_net,skill_count,posting_count,salary_numeric,salary_midpoint,salary_range_width,salary_comparable,posted_month
0,001514a00a1405d38498bba5e423836a50b64596d7a7e2...,1fa27923bb2775d71f12fe8b5915376b43749035958552...,ed28b9181a08d53d2625ee291e28cd9f64b08d21fb5613...,vietnamworks,SNOW VIỆT NAM,Zepeto Platform QA Tester (Tieng Han),QA Tester,QA / Testing,Mid-level,Ho Chi Minh,...,None,unknown,unknown,0,1,False,NaN,NaN,False,2026-10
1,025a48065a3243dc3f9de4305c496ce081a33564062257...,5fb121c3795a387c4611463e06217f4df6f4e02367e88b...,ed28b9181a08d53d2625ee291e28cd9f64b08d21fb5613...,vietnamworks,Tập đoàn Bưu chính Viễn thông Việt Nam (VNPT),Kỹ Sư Quản Lý Chất Lượng Sản Phẩm (QA),Quality Assurance Engineer,QA / Testing,Mid-level,Ha Noi,...,None,unknown,unknown,0,1,False,NaN,NaN,False,2026-10
2,02ad51f95ac987d039885406815323c8b680c57b11d0d0...,0c3aca2eb3b1bfb2731d798320ab490537553e3dea5a53...,ed28b9181a08d53d2625ee291e28cd9f64b08d21fb5613...,linkedin,Unknown,Quality Engineer,Quality Engineer,QA / Testing,Mid-level,Ho Chi Minh,...,None,unknown,unknown,1,1,False,NaN,NaN,False,None
3,02d97478e2f2d0047ac279bd07094f719f8de21a9cbd2e...,26e520db1c473e673405189fb175037c7d71277291e41a...,ed28b9181a08d53d2625ee291e28cd9f64b08d21fb5613...,topcv,CÔNG TY CỔ PHẦN TẬP ĐOÀN TRUYỀN THÔNG HALOTIMES,Chuyên Viên QA/QC TESTER (MANUAL),QA/QC Tester,QA / Testing,Junior,Ho Chi Minh,...,None,unknown,unknown,2,1,False,NaN,NaN,False,None
4,03871f50c0aa4efa1a0b51f65bd3bd88f8359d321cbce5...,a26181abb9fcea6ca81fe3b4a7bcf4f4f4f9296efce31a...,ed28b9181a08d53d2625ee291e28cd9f64b08d21fb5613...,linkedin,Unknown,Test Engineer (Fabric Kusto),Test Engineer,QA / Testing,Junior,Other,...,None,unknown,unknown,6,1,False,NaN,NaN,False,None


,dtype
posting_id,object
snapshot_id,object
version_id,object
source,object
company,object
title,object
title_cleaned,object
role,object
seniority,object
location,object


,field,reason,count
0,company,missing,625
1,date_posted,missing,1660
2,date_posted,unparsed,625
3,salary,missing,906
4,salary,unparsed,3
5,seniority,unknown,108
6,tech_stack,no_extracted_skills,742
7,work_model,unknown,764


,posting_id,source,salary_raw,salary_status,salary_min,salary_max,currency,pay_period,gross_net,date_raw,posted_date,tech_stack_raw,tech_stack_clean
0,193abd6e7ae06fb82e74cb5794eb7ded573315720f9b41...,itviec,Sign in to view salary,hidden,NaN,NaN,NaN,unknown,unknown,2026-10-03,2026-10-03,"[""Playwright"", ""Cypress"", ""Git"", ""GitHub"", ""AWS""]","[""AWS"", ""Cypress"", ""Git"", ""GitHub"", ""Playwright""]"
1,af9ee3427723578b8bc2e7e9d32f6c9d5a079f8d24efc1...,itviec,Sign in to view salary,hidden,NaN,NaN,NaN,unknown,unknown,2026-10-14,2026-10-14,"[""Python"", ""PowerShell"", ""Go"", ""C/C++"", ""Cobal...","[""Ansible"", ""Burp Suite"", ""C/C++"", ""Cobalt Str..."
2,59cc3ad763957a9cea52c5d4d47695e68e7b1ce874ee75...,itviec,Sign in to view salary,hidden,NaN,NaN,NaN,unknown,unknown,2026-10-09,2026-10-09,"[""Linux"", ""RHEL"", ""CentOS"", ""Ubuntu"", ""Nginx"",...","["".NET"", ""Apache"", ""CentOS"", ""Docker"", ""Java"",..."
3,22fc7ffc1f92a89cd07bb3bb7a919184ac40d6bbcc86c1...,itviec,Sign in to view salary,hidden,NaN,NaN,NaN,unknown,unknown,2026-10-09,2026-10-09,"[""Python"", ""Docker"", ""Jenkins"", ""GitLab CI"", ""...","[""Bash"", ""C++"", ""Docker"", ""GitHub Actions"", ""G..."
4,b92afb8ec6f6542fa918b44e8500faf96cce5a038524d3...,itviec,Sign in to view salary,hidden,NaN,NaN,NaN,unknown,unknown,2026-10-03,2026-10-03,"[""AWS"", ""Terraform"", ""Docker"", ""Linux"", ""Bash""...","[""AWS"", ""Bash"", ""Docker"", ""EC2"", ""ECS Fargate""..."
5,4f76ca3087c927d998a6887d0c356bf71bf55f76059f27...,itviec,Sign in to view salary,hidden,NaN,NaN,NaN,unknown,unknown,2026-10-14,2026-10-14,"[""SAP PP"", ""SAP S/4HANA"", ""SAP MM"", ""SAP WM"", ...","[""SAP FI/CO"", ""SAP MM"", ""SAP PP"", ""SAP QM"", ""S..."
6,fac188cacf06550d23b2dff9b5ace28fb42a80b2178aea...,itviec,Sign in to view salary,hidden,NaN,NaN,NaN,unknown,unknown,2026-10-14,2026-10-14,"[""Kubernetes"", ""Linux"", ""Python"", ""Bash"", ""Pow...","[""Ansible"", ""ArgoCD"", ""Azure AD"", ""Bash"", ""Key..."
7,7aa4aa8c11829278ec77de39c8b1fd375b039f2c8b723b...,itviec,Sign in to view salary,hidden,NaN,NaN,NaN,unknown,unknown,2026-10-17,2026-10-17,"[""Java"", ""Java EE"", ""SQL"", ""Oracle"", ""JUnit"", ...","[""Cassandra"", ""Java"", ""Java EE"", ""JUnit"", ""Kaf..."
8,c521afaad11b797af184e1fc34001a7c26575587983d43...,itviec,Sign in to view salary,hidden,NaN,NaN,NaN,unknown,unknown,2026-10-14,2026-10-14,[],[]
9,9b50f3322ac2e55ecccccb7c426ac07dc5b971bacaf0dc...,itviec,Sign in to view salary,hidden,NaN,NaN,NaN,unknown,unknown,2026-10-14,2026-10-14,"[""Kafka""]","[""Kafka""]"


## 2. Cơ cấu nguồn và nhóm nghề
Số tin theo nguồn đo độ đóng góp vào mẫu, không phải thị phần của website.

In [3]:
sources = df.source.value_counts().rename_axis('Nguồn').reset_index(name='Số tin')
px.bar(sources, x='Nguồn', y='Số tin', text='Số tin', title='Số tin theo nguồn').show()
roles = df.role.value_counts().rename_axis('Nhóm nghề').reset_index(name='Số tin')
px.bar(roles.sort_values('Số tin'), x='Số tin', y='Nhóm nghề', orientation='h', title='Cơ cấu nhóm nghề').show()
display(Markdown(f"Nguồn **{sources.iloc[0,0]}** chiếm **{sources.iloc[0,1]/len(df):.1%}** mẫu. Nhóm nghề đứng đầu là **{roles.iloc[0,0]}**. Kết quả phụ thuộc cơ cấu nguồn và nhãn phân loại kế thừa."))

Nguồn **topcv** chiếm **38.1%** mẫu. Nhóm nghề đứng đầu là **Data / AI / ML**. Kết quả phụ thuộc cơ cấu nguồn và nhãn phân loại kế thừa.

## 3. Địa điểm và tháng đăng
Địa điểm là nhãn chính do dự án gốc gán. `Other` có thể gồm nhiều địa điểm, ngoài ba thành phố chính hoặc thông tin chưa đủ. Tháng đăng chỉ mô tả mẫu có ngày, không chứng minh xu hướng nhu cầu tuyển dụng.

In [4]:
locations = df.location.value_counts().rename_axis('Địa điểm').reset_index(name='Số tin')
px.bar(locations, x='Địa điểm', y='Số tin', title='Phân bố địa điểm').show()
months = df.dropna(subset=['posted_month']).groupby('posted_month').size().reset_index(name='Số tin')
px.bar(months, x='posted_month', y='Số tin', labels={'posted_month':'Tháng đăng'}, title='Tháng đăng của các tin quan sát').show()
display(Markdown(f"Có **{df.posted_date.notna().sum():,}/{len(df):,}** tin có ngày hợp lệ; thiếu **{df.posted_date.isna().mean():.1%}**."))

Có **1,184/3,469** tin có ngày hợp lệ; thiếu **65.9%**.

## 4. Kỹ năng và quan hệ với nhóm nghề
Sử dụng `extracted_info.tech_stack`, không trộn kỹ năng nguồn với từ khóa AI. Tỷ lệ = số tin chứa kỹ năng / số tin trong nhóm. Một tin nhiều kỹ năng nên tổng tỷ lệ có thể vượt 100%.

In [5]:
top = skill_counts(df, tech).head(15)
display(top)
px.bar(top.sort_values('postings'), x='postings', y='skill', orientation='h', labels={'postings':'Số tin', 'skill':'Kỹ năng'}, title='Top 15 kỹ năng').show()
pairs = tech.merge(df[['snapshot_id','role']], on='snapshot_id').drop_duplicates(['snapshot_id','skill'])
heat = pd.crosstab(pairs.role, pairs.skill).reindex(columns=top.skill, fill_value=0)
heat = heat.div(df.groupby('role').size(), axis=0).mul(100).fillna(0)
px.imshow(heat, aspect='auto', color_continuous_scale='Blues', title='Tỷ lệ tin chứa kỹ năng trong từng nhóm nghề (%)').show()
display(Markdown(f"Kỹ năng xuất hiện nhiều nhất là **{top.iloc[0]['skill']}**, có trong **{int(top.iloc[0].postings)}** tin. Đây là tần suất trong mẫu, không phải xác suất xin được việc."))

,skill,postings,share_pct
1448,Python,912,26.289997
1641,SQL,533,15.364658
566,Docker,475,13.692707
913,Java,474,13.663880
1404,PostgreSQL,413,11.905448
960,Kubernetes,378,10.896512
1184,MySQL,300,8.648025
916,JavaScript,293,8.446238
1508,React,253,7.293168
1871,TypeScript,247,7.120208


Kỹ năng xuất hiện nhiều nhất là **Python**, có trong **912** tin. Đây là tần suất trong mẫu, không phải xác suất xin được việc.

## 5. Cấp bậc và mô hình làm việc
Mỗi tin có một nhãn cấp bậc chính; tin tuyển nhiều cấp bậc có thể bị giản lược ở dữ liệu nguồn. Tỷ lệ Remote bên dưới dùng toàn bộ tin làm mẫu số, giữ Unknown.

In [6]:
levels = df.seniority.value_counts().rename_axis('Cấp bậc').reset_index(name='Số tin')
px.bar(levels, x='Cấp bậc', y='Số tin', title='Cơ cấu cấp bậc').show()
work = pd.crosstab(df.role, df.work_model, normalize='index').mul(100).reset_index().melt(id_vars='role', var_name='Mô hình', value_name='Tỷ lệ (%)')
px.bar(work, x='role', y='Tỷ lệ (%)', color='Mô hình', labels={'role':'Nhóm nghề'}, title='Mô hình làm việc theo nhóm nghề').show()
remote = df.work_model.eq('Remote').sum()
known = df.work_model.ne('Unknown').sum()
display(Markdown(f"Remote: **{remote/len(df):.1%}** toàn mẫu; **{remote/known:.1%}** trong các tin biết mô hình. Hai tỷ lệ có mẫu số khác nhau."))

Remote: **3.1%** toàn mẫu; **4.0%** trong các tin biết mô hình. Hai tỷ lệ có mẫu số khác nhau.

## 6. Lương quảng cáo: điều kiện so sánh
Không tự coi `15–25 triệu` là lương theo tháng. Không tự đổi USD, năm, giờ sang VND/tháng. Từ X hoặc tới Y chỉ có một cận, không tính trung điểm. Thỏa thuận và bị ẩn không phải 0.

Trung điểm = (cận dưới + cận trên)/2, không phải lương thực nhận. Chỉ so sánh VND, kỳ tháng ghi rõ, đủ cận; tách gross/net/unknown.

In [7]:
display(pd.crosstab(df.salary_status, df.pay_period))
money = df[df.salary_comparable & df.gross_net.eq('unknown')].copy()
money['midpoint_million'] = money.salary_midpoint / 1_000_000
display(money.groupby('role').agg(n=('posting_id','size'), median_vnd=('salary_midpoint','median')))
if not money.empty:
    px.box(money, x='role', y='midpoint_million', points='all', labels={'role':'Nhóm nghề','midpoint_million':'Trung điểm (triệu VND/tháng)'}, title=f'Lương VND/tháng, không rõ gross/net — n={len(money)}').show()
display(Markdown(f"Có **{len(money)}** tin đáp ứng điều kiện của biểu đồ. Không suy rộng kết quả này cho các tin lương ẩn hoặc không ghi kỳ trả lương."))

pay_period,month,unknown,year
salary_status,,,
hidden,0,649,0
lower_bound,5,12,1
missing,0,906,0
negotiable,0,1086,0
range,97,560,8
unpaid,0,4,0
unparsed,3,0,0
upper_bound,28,108,2


,n,median_vnd
role,,
Backend,6,36500000.0
Data / AI / ML,18,35000000.0
DevOps / SRE / Cloud,1,17500000.0
Frontend,2,23250000.0
Fullstack,2,32500000.0
Infrastructure / IT Support,13,20000000.0
Other,5,27500000.0
Product / BA / Agile,2,47500000.0
QA / Testing,6,19250000.0


Có **56** tin đáp ứng điều kiện của biểu đồ. Không suy rộng kết quả này cho các tin lương ẩn hoặc không ghi kỳ trả lương.

## 7. Chất lượng dữ liệu và ứng viên trùng
Ứng viên được tìm bằng cùng công ty chuẩn hóa và tiêu đề tương đồng ≥ 0,85. Đây là quy tắc tìm cặp để kiểm tra, chưa được hiệu chỉnh bằng tập gán nhãn, không phải bằng chứng trùng. Không tự gộp.

In [8]:
q = quality(df)
display(q)
px.imshow(q, aspect='auto', zmin=0, zmax=100, text_auto=True, color_continuous_scale='YlOrBr', title='Tỷ lệ thiếu/chưa xác định theo nguồn (%)').show()
display(candidates[['left_title','right_title','score','decision']])
display(Markdown(f"Có **{len(candidates)}** cặp nghi trùng cần kiểm tra. Số tin trong các biểu đồ vẫn là số posting theo nguồn."))

,Thiếu ngày đăng hợp lệ,Không có lương số,Mô hình làm việc Unknown,Không có kỹ năng trích xuất
source,,,,
itviec,0.0,100.0,0.2,9.4
linkedin,100.0,100.0,45.8,16.0
topcv,100.0,48.6,1.4,21.9
topdev,89.9,100.0,23.7,50.1
vietnamworks,0.0,76.1,62.4,19.0


,left_title,right_title,score,decision
0,Chuyên Gia Phân Tích Dữ Liệu Khách Hàng Và Đán...,Chuyên Gia Phân Tích Dữ Liệu Khách Hàng Và Đán...,1.0000,pending
1,AI Engineer (hands-on),AI Engineer (Hands-On),1.0000,pending
2,Senior Security Architect,Senior Security Architect,1.0000,pending
3,Senior Odoo Software Engineer (Python),Senior Odoo Software Engineer,0.8923,pending
4,IT Operations & Support Lead,IT Operations & Support Lead,1.0000,pending
5,System Security Engineer (Senior),System Security Engineer (Senior),1.0000,pending
6,Database Engineer (DBA),Database Engineer (DBA),1.0000,pending
7,Delivery/Team Lead – Digitalization & Technolo...,Delivery/team Lead Digitalization & Technology...,0.9623,pending
8,Core Systems Center Director,Core Systems Center Director,1.0000,pending
9,Chuyên Gia Bảo Mật Thông Tin/An Ninh Mạng,Chuyên Gia Bảo Mật Thông Tin/An Ninh Mạng,1.0000,pending


Có **18** cặp nghi trùng cần kiểm tra. Số tin trong các biểu đồ vẫn là số posting theo nguồn.

## 8. Kết luận mô tả và giới hạn
Các nhận xét dưới đây được tính từ CSDL, không viết sẵn số liệu. Dữ liệu gốc đã được AI viết lại mô tả và phân loại. Bộ lọc tech gốc có thể loại BA/PM/UI/UX; không đại diện tất cả nghề IT. Chưa kiểm chứng nhãn AI, chưa gộp liên nguồn, chưa đánh giá toàn bộ lỗi trích xuất.

In [9]:
for line in insights(df, tech):
    display(Markdown('- ' + line))
df.to_csv(root / 'artifacts/analysis_postings.csv', index=False, encoding='utf-8-sig')
(root / 'artifacts/insights.md').write_text('# Nhận xét từ phiên bản dữ liệu ' + version['id'][:10] + '\n\n' + '\n\n'.join('- ' + line for line in insights(df, tech)), encoding='utf-8')

- Mẫu có 3,469 tin đăng. Nguồn topcv chiếm 38.1%; cơ cấu tổng thể chịu ảnh hưởng của nguồn này.

- Nhóm nghề Data / AI / ML đứng đầu mẫu với 626 tin (18.0%), dựa trên nhãn AI kế thừa.

- Python xuất hiện ở 912 tin (26.3% toàn bộ mẫu). Không có kỹ năng trích xuất chưa chắc là không yêu cầu kỹ năng.

- Remote chiếm 3.1% trên toàn bộ tin; 22.0% tin chưa rõ mô hình làm việc.

- 23.7% tin có lương số đọc được. Chỉ 56 tin có đủ hai cận VND và ghi rõ kỳ tháng; cần tiếp tục tách gross/net/unknown.

538